# 9-Qubit Shor Code in MLIR (Python bindings)

This notebook builds an **MLIR module** for the 9-qubit Shor encode/decode circuit using **Python MLIR bindings** (no full LLVM build).

We keep the IR portable by representing quantum gates as **external function calls** (`@h`, `@cx`, etc.) inside standard MLIR dialects (`func`, `tensor`, `arith`).

You can later swap these externs for a real quantum runtime / dialect lowering.


In [3]:
%pip -q install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install mlir numpy jupyter



Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# If you opened this notebook without installing deps, uncomment: 
# %pip -q install -r ../requirements.txt

import numpy as np

from src.shor_mlir import build_shor_module, canonicalize_and_cse


RuntimeError: Failed to import MLIR python bindings.
Try: `pip install mlir` (and restart kernel).
Original import error: cannot import name 'ir' from 'mlir' (c:\Users\vmuno\OneDrive\Desktop\CDAC\CDAC\Lib\site-packages\mlir\__init__.py)

## 1) Build MLIR for Shor encode/decode

We construct a module with:
- `func.func private @h(%q:i1)->i1`
- `func.func private @cx(%c:i1,%t:i1)->(i1,i1)`
- `func.func @shor_encode(%psi:i1)->tensor<9xi1>`
- `func.func @shor_decode(%code:tensor<9xi1>)->i1`

Here `i1` is just a placeholder type for a qubit handle; the goal is to get a clean MLIR graph you can later lower/translate.


In [ ]:
m = build_shor_module()
print(m)


## 2) Run a small pass pipeline

We run `canonicalize`, `cse`, and `symbol-dce` just to show the standard Python pass manager flow.


In [ ]:
canonicalize_and_cse(m)
print(m)


## 3) (Optional) Sanity check with a tiny NumPy simulator

This is **not** executing the MLIR. It just verifies the **encode/decode structure** matches the usual Shor circuit (encode then inverse-decode = identity on the logical qubit when no error is applied).

We simulate gates directly on a 9-qubit statevector (dimension 512), using the same gate order as in the MLIR builder.


In [ ]:
def kron_n(*ops):
    out = ops[0]
    for op in ops[1:]:
        out = np.kron(out, op)
    return out

I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)
H = (1/np.sqrt(2)) * np.array([[1,1],[1,-1]], dtype=complex)

def apply_1q(state, U, q, n=9):
    # q=0 is the most-significant qubit in this convention
    ops = [I]*n
    ops[q] = U
    Ufull = kron_n(*ops)
    return Ufull @ state

def apply_cx(state, c, t, n=9):
    # Build CX as projector method
    P0 = np.array([[1,0],[0,0]], dtype=complex)
    P1 = np.array([[0,0],[0,1]], dtype=complex)
    ops0 = [I]*n
    ops1 = [I]*n
    ops0[c] = P0
    ops1[c] = P1
    U0 = kron_n(*ops0)
    ops1[t] = X
    U1 = kron_n(*ops1)
    return (U0 + U1) @ state

def shor_encode_state(state):
    # same sequence as MLIR builder
    state = apply_cx(state, 0, 3)
    state = apply_cx(state, 0, 6)
    for q in (0,3,6):
        state = apply_1q(state, H, q)
    state = apply_cx(state, 0, 1)
    state = apply_cx(state, 0, 2)
    state = apply_cx(state, 3, 4)
    state = apply_cx(state, 3, 5)
    state = apply_cx(state, 6, 7)
    state = apply_cx(state, 6, 8)
    return state

def shor_decode_state(state):
    # inverse sequence
    state = apply_cx(state, 6, 8)
    state = apply_cx(state, 6, 7)
    state = apply_cx(state, 3, 5)
    state = apply_cx(state, 3, 4)
    state = apply_cx(state, 0, 2)
    state = apply_cx(state, 0, 1)
    for q in (0,3,6):
        state = apply_1q(state, H, q)
    state = apply_cx(state, 0, 6)
    state = apply_cx(state, 0, 3)
    return state

def basis(n, bits_int):
    v = np.zeros((2**n,), dtype=complex)
    v[bits_int] = 1.0
    return v

# Prepare |psi> on q0 and |0...0> on others.
# We'll test with |0> and |1>, and a random superposition.
zero9 = basis(9, 0)

def init_logical(alpha, beta):
    # state = (alpha|0> + beta|1>) \otimes |0..0>
    # With q0 as MSB, |1> on q0 corresponds to basis index 2^(8) = 256
    return alpha * basis(9, 0) + beta * basis(9, 256)

tests = [
    (1.0+0j, 0.0+0j),
    (0.0+0j, 1.0+0j),
    (0.37+0.12j, 0.81-0.44j),
]

for a,b in tests:
    # normalize
    nrm = np.sqrt(abs(a)**2 + abs(b)**2)
    a, b = a/nrm, b/nrm

    s0 = init_logical(a,b)
    s1 = shor_encode_state(s0)
    s2 = shor_decode_state(s1)

    # Compare reduced logical amplitudes (q0) by reading the |0..0> and |1 0..0> components.
    a2 = s2[0]
    b2 = s2[256]
    phase = a2/a if abs(a) > 1e-9 else b2/b

    ok = np.allclose(a2, phase*a, atol=1e-8) and np.allclose(b2, phase*b, atol=1e-8)
    print('PASS' if ok else 'FAIL', ' | recovered up to global phase')


## 4) Noise Model and Error Correction Testing

The Shor code can correct **one bit-flip error** OR **one phase-flip error** (but not both simultaneously in the same block). Let's test error correction with a noise model.


In [ ]:
from src.noise_model import NoiseModel, DepolarizingNoise, ErrorType

# Initialize noise model with 10% error rate
noise = NoiseModel(error_rate=0.1, seed=42)


### Test 1: Single Bit-Flip Error Correction

Let's test if the Shor code can correct a single bit-flip error on one of the physical qubits.


In [ ]:
# Test with |1> logical state
alpha, beta = 0.0, 1.0
s0 = init_logical(alpha, beta)

# Encode
s1 = shor_encode_state(s0)

# Apply a single bit-flip error on qubit 1 (should be correctable)
s1_noisy = noise.apply_single_error(s1, qubit=1, error_type=ErrorType.X)

# Decode
s2 = shor_decode_state(s1_noisy)

# Check if we recovered the original logical state
a2 = s2[0]
b2 = s2[256]

# For |1>, we expect b2 ≈ 1 (up to global phase)
recovered = abs(b2) > 0.99
print(f"Original: |1>")
print(f"After encode + X error on q1 + decode: amplitude on |1> = {b2:.6f}")
print(f"Error correction: {'SUCCESS ✓' if recovered else 'FAILED ✗'}")


### Test 2: Single Phase-Flip Error Correction

Now test phase-flip error correction.


In [ ]:
# Test with |+> = (|0> + |1>)/√2 logical state
alpha, beta = 1.0/np.sqrt(2), 1.0/np.sqrt(2)
s0 = init_logical(alpha, beta)

# Encode
s1 = shor_encode_state(s0)

# Apply a single phase-flip error on qubit 0 (should be correctable)
s1_noisy = noise.apply_single_error(s1, qubit=0, error_type=ErrorType.Z)

# Decode
s2 = shor_decode_state(s1_noisy)

# Check if we recovered the original logical state
a2 = s2[0]
b2 = s2[256]

# For |+>, we expect a2 ≈ b2 ≈ 1/√2
recovered = np.allclose([abs(a2), abs(b2)], [1/np.sqrt(2), 1/np.sqrt(2)], atol=0.1)
print(f"Original: |+> = (|0> + |1>)/√2")
print(f"After encode + Z error on q0 + decode:")
print(f"  Amplitude on |0>: {a2:.6f}")
print(f"  Amplitude on |1>: {b2:.6f}")
print(f"Error correction: {'SUCCESS ✓' if recovered else 'FAILED ✗'}")


### Test 3: Multiple Errors (Should Fail)

The Shor code can only correct **one error per block**. Let's see what happens with multiple errors.
yub

In [ ]:
# Test with |1> logical state
alpha, beta = 0.0, 1.0
s0 = init_logical(alpha, beta)

# Encode
s1 = shor_encode_state(s0)

# Apply TWO bit-flip errors in the same block (q0 and q1) - should fail
s1_noisy = noise.apply_single_error(s1, qubit=0, error_type=ErrorType.X)
s1_noisy = noise.apply_single_error(s1_noisy, qubit=1, error_type=ErrorType.X)

# Decode
s2 = shor_decode_state(s1_noisy)

# Check recovery
a2 = s2[0]
b2 = s2[256]

recovered = abs(b2) > 0.99
print(f"Original: |1>")
print(f"After encode + X errors on q0 AND q1 + decode: amplitude on |1> = {b2:.6f}")
print(f"Error correction: {'SUCCESS ✓' if recovered else 'FAILED ✗ (expected for 2 errors in same block)'}")


### Test 4: Random Error Statistics

Let's run many trials with random errors and see the success rate.


In [ ]:
def test_error_correction(initial_state_func, noise_model, n_trials=100, max_errors=1):
    """Test error correction over many trials."""
    successes = 0
    
    for trial in range(n_trials):
        # Prepare initial state
        s0 = initial_state_func()
        
        # Encode
        s1 = shor_encode_state(s0)
        
        # Apply random errors
        s1_noisy, errors_applied = noise_model.apply_random_errors(s1, max_errors=max_errors)
        
        # Decode
        s2 = shor_decode_state(s1_noisy)
        
        # Check if logical state is preserved (up to global phase)
        # For |1>, check if |1> component is still dominant
        a2 = s2[0]
        b2 = s2[256]
        
        # Simple check: if original was |1>, recovered should have |b2| > 0.9
        # For more general states, we'd need to compare full state
        if abs(b2) > 0.9:
            successes += 1
    
    return successes / n_trials

# Test with |1> state
print("Testing error correction with random single errors...")
print(f"Using noise model with error_rate={noise.error_rate}")

# Test with exactly 1 error
noise_single = NoiseModel(error_rate=1.0, seed=42)  # Always apply exactly 1 error
success_rate = test_error_correction(
    lambda: init_logical(0.0, 1.0),
    noise_single,
    n_trials=50,
    max_errors=1
)

print(f"\nSuccess rate (1 error): {success_rate*100:.1f}%")
print(f"Expected: ~100% (Shor code corrects single errors)")
